# Data Profiling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Libraries loaded successfully")

In [ ]:
from pathlib import Path

file_name = Path("../data/raw/online_retail_II.xlsx")
if not file_name.is_file():
    raise FileNotFoundError(
        "Download online_retail_II.xlsx from UCI and place it in data/raw/."
    )

In [ ]:
df_2009_2010 = pd.read_excel(
    file_name,
    sheet_name="Year 2009-2010",
    engine="openpyxl"
)

df_2010_2011 = pd.read_excel(
    file_name,
    sheet_name="Year 2010-2011",
    engine="openpyxl"
)

print(df_2009_2010.shape)
print(df_2010_2011.shape)

print(
    "Total rows:",
    len(df_2009_2010) + len(df_2010_2011)
)

In [ ]:
print(df_2009_2010.columns.tolist())
print(df_2010_2011.columns.tolist())

In [ ]:
df_2009_2010.head()

In [ ]:
df_2010_2011.head()

In [ ]:
print(df_2009_2010.dtypes)

In [ ]:
print(df_2010_2011.dtypes)

Date Coverage

In [ ]:
print(
    "2009-2010:",
    df_2009_2010["InvoiceDate"].min(),
    "to",
    df_2009_2010["InvoiceDate"].max()
)

print(
    "2010-2011:",
    df_2010_2011["InvoiceDate"].min(),
    "to",
    df_2010_2011["InvoiceDate"].max()
)

In [ ]:
missing_2009 = pd.DataFrame({
    "Missing Count": df_2009_2010.isnull().sum(),
    "Missing Percentage": (df_2009_2010.isnull().sum() / len(df_2009_2010)) * 100
})

missing_2009

In [ ]:
missing_2010 = pd.DataFrame({
    "Missing Count": df_2010_2011.isnull().sum(),
    "Missing Percentage": (df_2010_2011.isnull().sum() / len(df_2010_2011)) * 100
})

missing_2010

Where customer Ids are specifically missing

In [ ]:
df_2009_2010[df_2009_2010["Customer ID"].isnull()].head()

In [ ]:
df_2010_2011[df_2010_2011["Customer ID"].isnull()].head()

Number of customer Ids and description missing

In [ ]:
print(
    "Missing Customer ID 2009-2010:",
    df_2009_2010["Customer ID"].isnull().sum()
)

print(
    "Missing Customer ID 2010-2011:",
    df_2010_2011["Customer ID"].isnull().sum()
)

In [ ]:
print(
    "Missing Description 2009-2010:",
    df_2009_2010["Description"].isnull().sum()
)

print(
    "Missing Description 2010-2011:",
    df_2010_2011["Description"].isnull().sum()
)

Positive vs negative quantities among missing customers

In [ ]:
for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:
    missing_cust = df[df["Customer ID"].isna()]

    print("\n", name)

    print(
        "Positive quantity:",
        (missing_cust["Quantity"] > 0).sum()
    )

    print(
        "Negative quantity:",
        (missing_cust["Quantity"] < 0).sum()
    )

    print(
        "Zero quantity:",
        (missing_cust["Quantity"] == 0).sum()
    )

Check zero-price records among missing customers

In [ ]:
for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:
    missing_cust = df[df["Customer ID"].isna()]

    print(
        name,
        "Missing-ID rows with Price = 0:",
        (missing_cust["Price"] == 0).sum()
    )

Check whether missing-ID invoices are cancellations

In [ ]:
for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:
    missing_cust = df[df["Customer ID"].isna()].copy()

    cancelled = missing_cust["Invoice"].astype(str).str.startswith("C")

    print(
        name,
        "Missing-ID rows with C invoices:",
        cancelled.sum()
    )

Check normal-looking missing-ID purchases

In [ ]:
for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:
    normal_missing = df[
        (df["Customer ID"].isna()) &
        (df["Quantity"] > 0) &
        (df["Price"] > 0)
    ]

    print(
        name,
        "Missing-ID rows with positive Quantity and positive Price:",
        len(normal_missing)
    )

Duplicate records

In [ ]:
for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:
    duplicate_count = df.duplicated().sum()
    duplicate_percentage = duplicate_count / len(df) * 100

    print(
        name,
        "Duplicate rows:",
        duplicate_count,
        f"({duplicate_percentage:.2f}%)"
    )

In [ ]:
df_2009_2010[
    df_2009_2010.duplicated(keep=False)
].sort_values(
    by=["Invoice", "StockCode", "InvoiceDate"]
).head(20)

In [ ]:
df_2010_2011[
    df_2010_2011.duplicated(keep=False)
].sort_values(
    by=["Invoice", "StockCode", "InvoiceDate"]
).head(20)

Repeated product lines are distinct from exact duplicates. This project removes rows only when all original transaction fields are identical.

In [ ]:
for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:
    before = len(df)
    after = len(df.drop_duplicates())
    removed = before - after

    print(name)
    print("Rows before:", before)
    print("Rows after removing exact duplicates:", after)
    print("Rows removed:", removed)
    print()

In [ ]:
for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:
    dup_rows = df[df.duplicated(keep=False)]

    print("\n", name)

    print(
        "Duplicate rows with Customer ID:",
        dup_rows["Customer ID"].notna().sum()
    )

    print(
        "Duplicate rows with missing Customer ID:",
        dup_rows["Customer ID"].isna().sum()
    )

Cancellations and Negative Quantities

In [ ]:
for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:
    print("\n", name)

    print("Negative quantity rows:",
          (df["Quantity"] < 0).sum())

    print("Positive quantity rows:",
          (df["Quantity"] > 0).sum())

    print("Zero quantity rows:",
          (df["Quantity"] == 0).sum())

Then check invoices beginning with C

In [ ]:
for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:
    c_invoice = df["Invoice"].astype(str).str.startswith("C")

    print(
        name,
        "Rows with invoice starting C:",
        c_invoice.sum()
    )

are all C invoices negative?

In [ ]:
for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:
    c_rows = df[
        df["Invoice"].astype(str).str.startswith("C")
    ]

    print("\n", name)

    print("C rows with negative quantity:",
          (c_rows["Quantity"] < 0).sum())

    print("C rows with positive quantity:",
          (c_rows["Quantity"] > 0).sum())

    print("C rows with zero quantity:",
          (c_rows["Quantity"] == 0).sum())

Are there negative quantities that do not have a C invoice?

In [ ]:
for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:
    neg_rows = df[df["Quantity"] < 0]

    non_c_negative = neg_rows[
        ~neg_rows["Invoice"].astype(str).str.startswith("C")
    ]

    print(
        name,
        "Negative quantity rows without C invoice:",
        len(non_c_negative)
    )

In [ ]:
df_2009_2010[
    (df_2009_2010["Quantity"] < 0) &
    (~df_2009_2010["Invoice"].astype(str).str.startswith("C"))
].head(20)

In [ ]:
df_2010_2011[
    (df_2010_2011["Quantity"] < 0) &
    (~df_2010_2011["Invoice"].astype(str).str.startswith("C"))
].head(20)

In [ ]:
df_2009_2010[
    (df_2009_2010["Invoice"].astype(str).str.startswith("C")) &
    (df_2009_2010["Quantity"] > 0)
]

In [ ]:
for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:

    non_c_negative = df[
        (df["Quantity"] < 0) &
        (~df["Invoice"].astype(str).str.startswith("C"))
    ]

    print("\n", name)

    print("Total non-C negative rows:",
          len(non_c_negative))

    print("Price = 0:",
          (non_c_negative["Price"] == 0).sum())

    print("Price > 0:",
          (non_c_negative["Price"] > 0).sum())

    print("Missing Customer ID:",
          non_c_negative["Customer ID"].isna().sum())

    print("Has Customer ID:",
          non_c_negative["Customer ID"].notna().sum())

In [ ]:
for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:

    non_c_negative = df[
        (df["Quantity"] < 0) &
        (~df["Invoice"].astype(str).str.startswith("C"))
    ]

    print("\n", name)

    print(
        non_c_negative["Description"]
        .value_counts(dropna=False)
        .head(20)
    )

**Investigate Quantity Values**-Are very large quantities errors, or could they be legitimate bulk orders?

In [ ]:
for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:
    print("\n", name)
    print(df["Quantity"].describe())

check the largest positive quantities:

In [ ]:
df_2009_2010.nlargest(
    20,
    "Quantity"
)[[
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "Price",
    "Customer ID",
    "Country"
]]

In [ ]:
df_2010_2011.nlargest(
    20,
    "Quantity"
)[[
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "Price",
    "Customer ID",
    "Country"
]]

check the most negative quantities too

In [ ]:
df_2009_2010.nsmallest(
    20,
    "Quantity"
)[[
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "Price",
    "Customer ID",
    "Country"
]]

In [ ]:
df_2010_2011.nsmallest(
    20,
    "Quantity"
)[[
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "Price",
    "Customer ID",
    "Country"
]]

**Investigate Price**

In [ ]:
for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:
    print("\n", name)
    print(df["Price"].describe())

count zero and negative prices:

In [ ]:
for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:
    print("\n", name)

    print("Price > 0:",
          (df["Price"] > 0).sum())

    print("Price = 0:",
          (df["Price"] == 0).sum())

    print("Price < 0:",
          (df["Price"] < 0).sum())

Highest prices

In [ ]:
df_2009_2010.nlargest(
    20, "Price"
)[[
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "Price",
    "Customer ID",
    "Country"
]]

In [ ]:
df_2010_2011.nlargest(
    20, "Price"
)[[
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "Price",
    "Customer ID",
    "Country"
]]

Zero price rows

In [ ]:
df_2009_2010[
    df_2009_2010["Price"] == 0
].head(20)

In [ ]:
df_2010_2011[
    df_2010_2011["Price"] == 0
].head(20)

In [ ]:
for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:
    print("\n", name)

    display(
        df[df["Price"] < 0][[
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country"
        ]]
    )

In [ ]:
for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:

    zero_price = df[df["Price"] == 0]

    print("\n", name)
    print("Total zero-price rows:", len(zero_price))

    print(
        "Zero-price rows with Customer ID:",
        zero_price["Customer ID"].notna().sum()
    )

    print(
        "Zero-price rows with missing Customer ID:",
        zero_price["Customer ID"].isna().sum()
    )

    print(
        "Zero-price rows with positive Quantity:",
        (zero_price["Quantity"] > 0).sum()
    )

    print(
        "Zero-price rows with negative Quantity:",
        (zero_price["Quantity"] < 0).sum()
    )

Price = 0 but Customer ID exists

In [ ]:
df_2009_2010[
    (df_2009_2010["Price"] == 0) &
    (df_2009_2010["Customer ID"].notna())
][[
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country"
]].head(30)

In [ ]:
df_2010_2011[
    (df_2010_2011["Price"] == 0) &
    (df_2010_2011["Customer ID"].notna())
][[
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country"
]].head(30)

**Investigate special StockCode values**

In [ ]:
for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:
    print("\n", name)

    print(
        df["StockCode"]
        .astype(str)
        .value_counts()
        .head(30)
    )

In [ ]:
for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:

    special = df[
        df["StockCode"]
        .astype(str)
        .str.contains(r"[A-Za-z]", regex=True)
    ]

    print("\n", name)
    print(
        special["StockCode"]
        .astype(str)
        .value_counts()
        .head(40)
    )

**targeted check**

In [ ]:
special_codes = [
    "POST",
    "M",
    "DOT",
    "AMAZONFEE",
    "B",
    "BANK CHARGES",
    "TEST001",
    "PADS"
]

for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:

    print("\n", name)

    special_rows = df[
        df["StockCode"]
        .astype(str)
        .isin(special_codes)
    ]

    print(
        special_rows["StockCode"]
        .astype(str)
        .value_counts()
    )

In [ ]:
for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:

    print("\n", name)

    special_rows = df[
        df["StockCode"]
        .astype(str)
        .isin(special_codes)
    ]

    display(
        special_rows[
            [
                "StockCode",
                "Description",
                "Quantity",
                "Price",
                "Customer ID"
            ]
        ]
        .drop_duplicates()
        .head(50)
    )

**Outlier Assesment**

In [ ]:
for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:
    print("\n", name)

    for col in ["Quantity", "Price"]:

        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        outliers = (
            (df[col] < lower) |
            (df[col] > upper)
        ).sum()

        print(
            col,
            "| Q1:", q1,
            "| Q3:", q3,
            "| IQR:", iqr,
            "| Lower:", lower,
            "| Upper:", upper,
            "| IQR outliers:", outliers
        )

In [ ]:
for name, df in [
    ("2009-2010", df_2009_2010),
    ("2010-2011", df_2010_2011)
]:
    print("\n", name)

    print(
        "Quantity skewness:",
        df["Quantity"].skew()
    )

    print(
        "Price skewness:",
        df["Price"].skew()
    )